# Grab data from UN Digital Library

The UNDL contains a large archive of historical voting data readily available without the need to do any web scraping. The first task will be to work with this data and assimilate it into our working dataset.

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

UN_DL_DIR = Path.cwd().parent / "un_dl"
GA_FILE = UN_DL_DIR / "2026_02_06_ga_voting.csv"
SC_FILE = UN_DL_DIR / "2026_02_06_sc_voting.csv"
MS_FILE = UN_DL_DIR / "member_states_auths_2025-12-02_rev-1.csv"

df = pd.read_csv(GA_FILE)
df = df[['undl_id', 'ms_code', 'ms_name', 'ms_vote', 'date', 'session', 'resolution', 'title', 'subjects']]

In [ ]:
df['title'] = [x.split(': resolution')[0].strip() for x in df['title']]

group_cols = ['undl_id', 'date', 'session', 'resolution', 'title', 'subjects']

def vote_records(group, accepted_votes):
    votes = group[group['ms_vote'].astype(str).str.strip().str.upper().isin(accepted_votes)]
    votes = votes[['ms_code', 'ms_name']].dropna(subset=['ms_code', 'ms_name']).drop_duplicates()
    return votes.to_dict(orient='records')

grouped_records = []
for keys, group in df.groupby(group_cols, dropna=False):
    record = dict(zip(group_cols, keys))
    record['yes_votes'] = vote_records(group, {'Y', 'YES'})
    record['no_votes'] = vote_records(group, {'N', 'NO'})
    record['abstain_votes'] = vote_records(group, {'A', 'ABSTAIN', 'ABSTENTION'})
    grouped_records.append(record)

df_grouped = pd.DataFrame(grouped_records, columns=[
    'undl_id',
    'date',
    'session',
    'resolution',
    'title',
    'subjects',
    'yes_votes',
    'no_votes',
    'abstain_votes',
])

df_grouped.head(100)
df_grouped.to_json(Path.cwd().parent / "live" / "ga_archive.json", orient="records", indent=4)